# Phase 2: Simulation Components & Baselines
**Objective:** Construct the dynamic logic modules that will interact with the SUMO engine.

## 1. Stochastic Incident Generation (Patel et al. 2016)
We generate a deterministic schedule of emergencies. The probability distribution is spatially weighted towards major traffic hotspots in Kigali (e.g., Nyabugogo, Giporoso) and utilizes an injury severity ratio consistent with local epidemiological research.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.environment.incident_gen import StochasticIncidentGenerator

net_path = Path("../data/processed/kigali.net.xml")
incidents_output_path = Path("../data/processed/incidents_seed42.json")

generator = StochasticIncidentGenerator(net_path=net_path, seed=42)

generator.generate(
    output_path=incidents_output_path,
    num_incidents=30,
    duration_seconds=3600
)

2026-03-24 10:40:03,208 - INFO - Generating 30 stochastic incidents (Seed: 42)...
2026-03-24 10:40:04,815 - INFO - Calculating spatial probability weights for network edges...
2026-03-24 10:40:05,124 - INFO - Successfully saved incident schedule to ../data/processed/incidents_seed42.json


In [2]:
from src.environment.hospital import Hospital, calculate_time_to_care

# Initialize the complete network of Kigali Referral and District Hospitals.
# Note: Capacities (ED bays) and service rates are estimated baselines for the simulation.
# edge_ids remain placeholders until we map the exact map coordinates in Phase 4.
hospitals = {
    "CHUK": Hospital("H_CHUK", "University Teaching Hospital of Kigali (CHUK)", "edge_chuk", capacity=20, service_rate_per_hour=1.5),
    "KFH": Hospital("H_KFH", "King Faisal Hospital (KFH)", "edge_kfh", capacity=10, service_rate_per_hour=2.0),
    "RMH": Hospital("H_RMH", "Rwanda Military Referral (RMH)", "edge_rmh", capacity=15, service_rate_per_hour=1.5),
    "KIBAGABAGA": Hospital("H_KIB", "Kibagabaga District Hospital", "edge_kib", capacity=8, service_rate_per_hour=1.2),
    "NYARUGENGE": Hospital("H_NYA", "Nyarugenge District Hospital", "edge_nya", capacity=8, service_rate_per_hour=1.2),
    "KACYIRU": Hospital("H_KAC", "Kacyiru District Hospital", "edge_kac", capacity=8, service_rate_per_hour=1.2),
    "MASAKA": Hospital("H_MAS", "Masaka District Hospital", "edge_mas", capacity=8, service_rate_per_hour=1.2),
    "MUHIMA": Hospital("H_MUH", "Muhima District Hospital", "edge_muh", capacity=6, service_rate_per_hour=1.2)
}

# --- Testing the Queuing Dynamics ---
print("--- Baseline State ---")
for key, hosp in hospitals.items():
    print(f"{key}: Queue={hosp.current_queue:.1f}, Wait Time={hosp.estimate_wait_time():.1f}s")

# Simulate a localized incident near Nyarugenge, instantly admitting 12 patients to the district hospital
print("\n--- After admitting 12 patients to Nyarugenge District Hospital ---")
for _ in range(12):
    hospitals["NYARUGENGE"].admit_patient()

wait_time_nya = hospitals['NYARUGENGE'].estimate_wait_time() / 60
print(f"Nyarugenge: Queue={hospitals['NYARUGENGE'].current_queue:.1f}, Wait Time={wait_time_nya:.1f} minutes")

# Compare Time-to-Care
# Scenario: An ambulance is 3 minutes (180s) from Nyarugenge, but 10 minutes (600s) from CHUK.
time_to_nya = calculate_time_to_care(travel_time_seconds=180, hospital=hospitals["NYARUGENGE"])
time_to_chuk = calculate_time_to_care(travel_time_seconds=600, hospital=hospitals["CHUK"])

print("\n--- Routing Decision ---")
print(f"Total Time-to-Care (Nyarugenge - 3 min drive): {time_to_nya/60:.1f} minutes")
print(f"Total Time-to-Care (CHUK - 10 min drive): {time_to_chuk/60:.1f} minutes")
print("Conclusion: The RL agent should bypass the physically closer Nyarugenge hospital to avoid the severe backlog.")

--- Baseline State ---
CHUK: Queue=0.0, Wait Time=0.0s
KFH: Queue=0.0, Wait Time=0.0s
RMH: Queue=0.0, Wait Time=0.0s
KIBAGABAGA: Queue=0.0, Wait Time=0.0s
NYARUGENGE: Queue=0.0, Wait Time=0.0s
KACYIRU: Queue=0.0, Wait Time=0.0s
MASAKA: Queue=0.0, Wait Time=0.0s
MUHIMA: Queue=0.0, Wait Time=0.0s

--- After admitting 12 patients to Nyarugenge District Hospital ---
Nyarugenge: Queue=12.0, Wait Time=31.2 minutes

--- Routing Decision ---
Total Time-to-Care (Nyarugenge - 3 min drive): 34.2 minutes
Total Time-to-Care (CHUK - 10 min drive): 10.0 minutes
Conclusion: The RL agent should bypass the physically closer Nyarugenge hospital to avoid the severe backlog.
